# RawFileReader integration test

This notebook exercises `RawFileAdapter` against the repository's real `sample.raw` and the bundled .NET 8 RawFileReader assemblies. It can be run directly in Google Colab or locally from either the repository root or the `tests` directory.

## Setup

The next cell installs `pythonnet` in every environment. On Google Colab, it also installs the .NET 8 runtime. Local environments must provide .NET 8 through their operating-system package manager.


In [ ]:
import importlib.util
import os
import shutil
import subprocess
import sys

IN_COLAB = importlib.util.find_spec("google.colab") is not None

if IN_COLAB and shutil.which("dotnet") is None:
    subprocess.run(
        [
            "bash",
            "-c",
            "set -e; "
            "wget -q https://packages.microsoft.com/config/ubuntu/22.04/"
            "packages-microsoft-prod.deb -O /tmp/packages-microsoft-prod.deb; "
            "dpkg -i /tmp/packages-microsoft-prod.deb >/dev/null; "
            "rm /tmp/packages-microsoft-prod.deb; "
            "apt-get update -qq; "
            "apt-get install -y -qq dotnet-runtime-8.0",
        ],
        check=True,
    )

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "pythonnet>=3.0.3"],
    check=True,
)
os.environ.setdefault("DOTNET_ROOT", "/usr/share/dotnet")

print("pythonnet installed")
print("DOTNET_ROOT:", os.environ["DOTNET_ROOT"])
print("dotnet executable:", shutil.which("dotnet") or "not found")


In [ ]:
from pathlib import Path

if IN_COLAB:
    repo_root = Path("/content/RawFileReaderPyAdapter")
    if not repo_root.is_dir():
        subprocess.run(
            [
                "git",
                "clone",
                "--depth",
                "1",
                "https://github.com/mzzzhunter/RawFileReaderPyAdapter.git",
                str(repo_root),
            ],
            check=True,
        )
else:
    working_directory = Path.cwd().resolve()
    repo_root = (
        working_directory.parent
        if working_directory.name == "tests"
        else working_directory
    )

if not (repo_root / "rawfilereader").is_dir():
    raise RuntimeError("Could not locate the RawFileReaderPyAdapter repository")

sys.path.insert(0, str(repo_root))

from rawfilereader import RawFileAdapter


In [ ]:
sample_raw = repo_root / "sample.raw"
assemblies = repo_root / "libs" / "Net8" / "Assemblies"
required_assemblies = (
    "OpenMcdf.dll",
    "OpenMcdf.Extensions.dll",
    "ThermoFisher.CommonCore.Data.dll",
    "ThermoFisher.CommonCore.RawFileReader.dll",
    "ThermoFisher.CommonCore.BackgroundSubtraction.dll",
)

assert sample_raw.is_file(), f"Missing integration fixture: {sample_raw}"
assert sample_raw.stat().st_size > 0, "sample.raw is empty"
for assembly_name in required_assemblies:
    assembly_path = assemblies / assembly_name
    assert assembly_path.is_file(), f"Missing assembly: {assembly_path}"

print(f"RAW fixture: {sample_raw}")
print(f"Assemblies: {assemblies}")


In [ ]:
adapter = RawFileAdapter(str(sample_raw), libs_dir=str(assemblies))
assert not adapter.is_open

with adapter:
    assert adapter.is_open
    first_scan, last_scan = adapter.get_scan_range()
    assert first_scan >= 1
    assert last_scan >= first_scan

    file_info = adapter.get_file_info()
    first_scan_data = adapter.get_centroid_stream(first_scan)

    available_instruments = []
    for device_type in adapter.DEVICE_TYPES:
        instrument_count = adapter.get_instrument_count_of_type(device_type)
        if instrument_count > 0:
            available_instruments.append((device_type, instrument_count))
    assert available_instruments, "No selectable instruments found in sample.raw"
    target_type, target_count = available_instruments[-1]
    target_instance = target_count
    adapter.select_instrument(target_type, target_instance)
    selected_instrument = adapter.get_instrument_data()
    assert selected_instrument.device_type == target_type
    assert selected_instrument.instance_number == target_instance

    print(f"File: {file_info.file_name}")
    print(f"Scan range: {first_scan}–{last_scan}")
    print(f"First scan centroid peaks: {len(first_scan_data.masses)}")
    print(f"Selected instrument: {target_type} instance {target_instance}")

assert not adapter.is_open
print("Integration test passed; the native RAW file was closed.")
